In [9]:
!git clone https://github.com/Manish-reddy0905/real-estate-price-predictor.git

fatal: destination path 'real-estate-price-predictor' already exists and is not an empty directory.


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
# =========================================================
# REAL ESTATE PRICE PREDICTOR API PREP PROJECT
# =========================================================

# Install required libraries (Run once in Colab)
!pip install pyarrow fastparquet seaborn -q

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from google.colab import files

# =========================================================
# UPLOAD CSV DATASET
# =========================================================

print("Upload your housing_transactions.csv file")

uploaded = files.upload()

# Get uploaded filename automatically
INPUT_FILE = list(uploaded.keys())[0]

OUTPUT_FILE = "cleaned_real_estate.parquet"

# =========================================================
# LOAD DATA IN CHUNKS
# =========================================================

print("\nLoading dataset in chunks...")

CHUNK_SIZE = 100000

chunks = []

for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):

    # Clean column names
    chunk.columns = chunk.columns.str.strip().str.lower()

    # Remove duplicate rows
    chunk.drop_duplicates(inplace=True)

    # Remove rows with missing prices
    chunk.dropna(subset=["price"], inplace=True)

    chunks.append(chunk)

# Combine all chunks
df = pd.concat(chunks, ignore_index=True)

print("\nDataset Loaded Successfully")
print("Dataset Shape:", df.shape)

# =========================================================
# EXPLORATORY DATA ANALYSIS (EDA)
# =========================================================

print("\n========== DATASET INFO ==========")
print(df.info())

print("\n========== STATISTICAL SUMMARY ==========")
print(df.describe())

print("\n========== MISSING VALUES ==========")
print(df.isnull().sum())

# =========================================================
# VISUALIZATION - HOUSE PRICE DISTRIBUTION
# =========================================================

plt.figure(figsize=(12,6))

sns.histplot(
    df["price"],
    bins=50,
    kde=True
)

plt.title("Distribution of House Prices")
plt.xlabel("House Price")
plt.ylabel("Frequency")

plt.show()

# =========================================================
# REMOVE OUTLIERS USING 3 STANDARD DEVIATIONS
# =========================================================

print("\nRemoving outliers...")

mean_price = df["price"].mean()
std_price = df["price"].std()

lower_limit = mean_price - (3 * std_price)
upper_limit = mean_price + (3 * std_price)

original_rows = len(df)

df = df[
    (df["price"] >= lower_limit) &
    (df["price"] <= upper_limit)
]

removed_rows = original_rows - len(df)

print(f"\nOutliers Removed: {removed_rows}")
print(f"Remaining Rows: {len(df)}")

# =========================================================
# FEATURE ENGINEERING
# =========================================================

print("\nEngineering New Features...")

CURRENT_YEAR = 2026

# ---------------------------------------------------------
# Feature 1: Age of Home
# ---------------------------------------------------------

if "year_built" in df.columns:
    df["age_of_home"] = CURRENT_YEAR - df["year_built"]

# ---------------------------------------------------------
# Feature 2: Price Per Square Foot
# ---------------------------------------------------------

if "square_feet" in df.columns:
    df["price_per_sqft"] = (
        df["price"] / df["square_feet"]
    )

# ---------------------------------------------------------
# Feature 3: Distance to City Center
# ---------------------------------------------------------

if {"latitude", "longitude"}.issubset(df.columns):

    # Example city center coordinates
    CITY_CENTER_LAT = 40.7128
    CITY_CENTER_LON = -74.0060

    df["distance_to_city_center"] = np.sqrt(
        (df["latitude"] - CITY_CENTER_LAT) ** 2 +
        (df["longitude"] - CITY_CENTER_LON) ** 2
    )

print("\nNew Features Added Successfully")

# =========================================================
# HANDLE MISSING VALUES
# =========================================================

print("\nHandling Missing Values...")

# Numeric columns → fill with median
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns → fill with mode
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# =========================================================
# OPTIMIZE DATA TYPES
# =========================================================

print("\nOptimizing Data Types...")

for col in df.select_dtypes(include="int64").columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")

for col in df.select_dtypes(include="float64").columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

# =========================================================
# SAVE CLEANED DATA TO PARQUET FORMAT
# =========================================================

print("\nSaving cleaned dataset to parquet format...")

df.to_parquet(
    OUTPUT_FILE,
    engine="pyarrow",
    index=False
)

print(f"\nParquet File Saved Successfully: {OUTPUT_FILE}")

# =========================================================
# DOWNLOAD PARQUET FILE
# =========================================================

files.download(OUTPUT_FILE)

# =========================================================
# FINAL OUTPUT
# =========================================================

print("\n========== PROJECT COMPLETED ==========")

print("Final Dataset Shape:", df.shape)

print("\nFinal Columns:")
print(df.columns.tolist())

print("\nProject Execution Successful!")

### Reading JSON Files
To read a JSON file, you can use `pd.read_json()`.

In [ ]:
# Example for reading a JSON file
# Assuming 'data.json' exists in your environment
# df_json = pd.read_json('data.json')
# display(df_json.head())

### Reading Excel Files
For Excel files (.xlsx, .xls), use `pd.read_excel()`.

In [ ]:
# Example for reading an Excel file
# Assuming 'data.xlsx' exists in your environment
# You might need to install openpyxl: !pip install openpyxl
# df_excel = pd.read_excel('data.xlsx')
# display(df_excel.head())

### Reading Parquet Files
Parquet is a columnar storage file format, often used for big data analytics. Use `pd.read_parquet()`.

In [ ]:
# Example for reading a Parquet file
# Assuming 'data.parquet' exists in your environment
# You might need to install pyarrow or fastparquet: !pip install pyarrow
# df_parquet = pd.read_parquet('data.parquet')
# display(df_parquet.head())

Remember to replace `'data.json'`, `'data.xlsx'`, or `'data.parquet'` with the actual path to your file, and ensure you have the necessary libraries installed for each format (e.g., `openpyxl` for Excel, `pyarrow` or `fastparquet` for Parquet).